# 08. Sentence-BERT 기반 HS 지식베이스 의미 검색

07번은 기존 검색 근거와 규칙 유형을 점검하는 단계였습니다. 08번에서는 다국어 문장 임베딩 모델을 이용해 제품 설명과 HS 지식 문서의 의미적 유사도를 계산하고, 기존 TF-IDF 검색과 비교합니다.

검색 품질 평가는 HS 코드가 명시된 지식 문서만 대상으로 합니다. 제품의 정답 HS 코드와 계층적으로 호환되는 문서를 관련 문서로 정의하며, 이 정답 정보는 평가 지표 계산에만 사용합니다. HS 코드가 없는 일반 규정은 관련 문서 정답을 정의하기 어려우므로 별도 검색 결과로 보존하고 별도 진단합니다.

이 실험은 분류 모델을 교체하거나 HS 코드를 자동 결정하지 않습니다. 검색 단계의 개선 여부를 검증하는 실험입니다.


In [ ]:
!pip -q install -U sentence-transformers


In [ ]:
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import normalize


In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
data_all = pd.read_csv(BASE+cases_file).fillna('')
kb = pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
input_cols = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
field_weights = {'Product Name':0.30,'Main Ingredients (KR)':0.20,'Main Ingredients (EN)':0.10,'Product Form and Packaging':0.20,'Intended Use':0.20}
data_all['model_text'] = data_all[input_cols].astype(str).agg(' '.join,axis=1)
data_all['duplicate_group'] = data_all[input_cols].astype(str).agg('||'.join,axis=1).map(lambda x:hashlib.md5(x.encode()).hexdigest())
eligible = data_all.groupby('HS Code').filter(lambda x:len(x)>=4)
group_counts = eligible.groupby('HS Code')['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts>=4].index.astype(str))
data = eligible[eligible['HS Code'].astype(str).isin(labels)].copy().reset_index(drop=True)
data['HS Code'] = data['HS Code'].astype(str)
kb['document_text'] = kb['document_text'].fillna('').astype(str)
kb['hs_code'] = kb['hs_code'].fillna('').astype(str)
print('제품:',len(data),'| HS 코드:',data['HS Code'].nunique(),'| 지식 문서:',len(kb))


## 1. 검색 평가 기준

HS 코드가 명시된 문서 중 문서 코드와 제품 정답 코드가 계층적으로 호환되는 문서를 관련 문서로 정의합니다. 예를 들어 한 코드가 다른 코드의 상위·하위 접두어 관계이면 호환으로 처리합니다. 이 정의는 검색 성능 비교를 위한 평가 규칙이며, 실제 법적 적용 가능성을 확정하지 않습니다.

- Recall@5 / Recall@10: 상위 k개 검색 문서 중 관련 문서가 하나 이상 포함되는 제품 비율
- MRR: 첫 번째 관련 문서의 순위 역수 평균
- nDCG@10: 상위 10개 결과에서 관련 문서가 얼마나 높은 순위에 배치되는지 평가

일반 규정처럼 HS 코드가 없는 문서는 위 지표의 관련 문서 정답을 만들 수 없으므로 별도 집계합니다.


In [ ]:
def code_digits(value): return re.sub(r'\\D','',str(value))
def code_compatible(doc_code,true_code):
    a,b = code_digits(doc_code),code_digits(true_code)
    return bool(a and b and (a.startswith(b) or b.startswith(a)))
kb['code_digits'] = kb['hs_code'].map(code_digits)
kb['is_code_specific'] = kb['code_digits'].ne('')
kb['search_text'] = (kb['document_text']+' '+kb['rule_type'].astype(str)+' '+kb['source_sheet'].astype(str)).str.strip()
print('코드 명시 문서:',int(kb['is_code_specific'].sum()),'| 일반 문서:',int((~kb['is_code_specific']).sum()))


## 2. TF-IDF 기준 검색기

기존 04~06번에서 사용한 단어 및 문자 n-gram 기반 TF-IDF를 기준선으로 재구현합니다. 전체 제품 설명을 하나의 질의로 사용하는 방식과 다섯 속성별 유사도를 가중합하는 방식을 비교합니다.


In [ ]:
tfidf = FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),max_features=30000,sublinear_tf=True))])
tfidf_docs = tfidf.fit_transform(kb['search_text'])
def tfidf_scores(query):
    query = str(query).strip()
    if not query: return np.zeros(len(kb))
    return (tfidf_docs @ tfidf.transform([query]).T).toarray().ravel()
def tfidf_attribute_scores(row):
    score = np.zeros(len(kb))
    for field,weight in field_weights.items(): score += weight*tfidf_scores(row[field])
    return score


## 3. Sentence-BERT 의미 검색기

한국어와 영어가 혼합된 제품명·성분·용도 및 지식 문서를 함께 처리하기 위해 다국어 E5 임베딩 모델 `intfloat/multilingual-e5-base`를 사용합니다. E5 모델의 입력 형식에 맞춰 문서에는 `passage:`, 검색 질의에는 `query:` 접두어를 붙입니다.

문서 임베딩은 한 번 계산하고 재사용합니다. 제품 전체 설명 검색과 속성별 가중 검색을 모두 구현합니다.


In [ ]:
MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME)
passages = ['passage: '+x for x in kb['search_text'].tolist()]
doc_embeddings = model.encode(passages,batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
def sbert_scores(query):
    query = str(query).strip()
    if not query: return np.zeros(len(kb))
    emb = model.encode(['query: '+query],normalize_embeddings=True,convert_to_numpy=True)
    return (doc_embeddings @ emb[0]).ravel()
def sbert_attribute_scores(row):
    score = np.zeros(len(kb))
    for field,weight in field_weights.items(): score += weight*sbert_scores(row[field])
    return score


## 4. 검색 성능 평가

동일한 4-fold StratifiedGroupKFold 분할을 사용해 평가 제품을 구성합니다. 검색 인덱스는 독립된 지식베이스로 구성되어 있으며, 정답 HS 코드는 관련 문서 판정과 지표 계산에만 사용합니다.

각 제품에 대해 TF-IDF 전체 질의, TF-IDF 속성별 가중 검색, Sentence-BERT 전체 질의, Sentence-BERT 속성별 가중 검색을 비교합니다.


In [ ]:
def ranking_metrics(scores,true_code,kb_frame=kb):
    order = np.argsort(scores)[::-1]
    relevant = np.array([code_compatible(x,true_code) for x in kb_frame['hs_code']]) & kb_frame['is_code_specific'].to_numpy()
    ranked_rel = relevant[order]
    hits = np.where(ranked_rel)[0]
    mrr = 1/(hits[0]+1) if len(hits) else 0.0
    recalls = {f'recall@{k}':float(ranked_rel[:k].any()) for k in [5,10]}
    gains = ranked_rel[:10].astype(float)
    discounts = 1/np.log2(np.arange(2,len(gains)+2))
    dcg = float((gains*discounts).sum())
    total_rel = int(relevant.sum())
    ideal_n = min(total_rel,10)
    idcg = float((np.ones(ideal_n)/np.log2(np.arange(2,ideal_n+2))).sum()) if ideal_n else 0.0
    return {'mrr':mrr,'ndcg@10':dcg/idcg if idcg else 0.0,**recalls,'relevant_doc_count':total_rel,'first_relevant_rank':int(hits[0]+1) if len(hits) else np.nan}
cv = StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=42)
methods = ['TFIDF_combined','TFIDF_attribute','SBERT_combined','SBERT_attribute']
result_rows, ranking_rows, fold_rows = [],[],[]
for fold,(tr,te) in enumerate(cv.split(data['model_text'],data['HS Code'],groups=data['duplicate_group']),1):
    train,test = data.iloc[tr],data.iloc[te]
    overlap = len(set(train['duplicate_group'])&set(test['duplicate_group']))
    fold_rows.append({'fold':fold,'train_n':len(tr),'test_n':len(te),'group_overlap':overlap})
    for _,row in test.iterrows():
        score_map = {'TFIDF_combined':tfidf_scores(row['model_text']),'TFIDF_attribute':tfidf_attribute_scores(row),'SBERT_combined':sbert_scores(row['model_text']),'SBERT_attribute':sbert_attribute_scores(row)}
        for method,scores in score_map.items():
            met = ranking_metrics(scores,row['HS Code'])
            result_rows.append({'fold':fold,'row_index':int(row.name),'true_hs':row['HS Code'],'method':method,**met})
            order = np.argsort(scores)[::-1][:10]
            for rank,kb_idx in enumerate(order,1):
                d = kb.iloc[kb_idx]
                ranking_rows.append({'fold':fold,'row_index':int(row.name),'true_hs':row['HS Code'],'method':method,'rank':rank,'knowledge_id':d['knowledge_id'],'source_sheet':d['source_sheet'],'rule_type':d['rule_type'],'rule_hs_code':d['hs_code'],'is_code_specific':bool(d['is_code_specific']),'is_relevant':code_compatible(d['hs_code'],row['HS Code']) if d['is_code_specific'] else False,'score':float(scores[kb_idx]),'document_text':d['document_text']})
    print('fold',fold,'완료')
results = pd.DataFrame(result_rows)
rankings = pd.DataFrame(ranking_rows)
fold_audit = pd.DataFrame(fold_rows)
display(fold_audit)


## 5. 모델 비교

제품 단위로 계산한 검색 지표의 평균과 표준편차를 비교합니다. 검색 결과는 제품별로 여러 문서가 연결될 수 있으므로 제품 단위 지표를 기준으로 요약합니다.


In [ ]:
metric_cols = ['mrr','ndcg@10','recall@5','recall@10']
comparison = results.groupby('method')[metric_cols].agg(['mean','std']).round(4)
display(comparison)
display(results.groupby('method')['relevant_doc_count'].agg(['mean','median','min','max']))
display(rankings.groupby(['method','rank'])['is_relevant'].mean().unstack(0).round(3))


## 6. 일반 규정 검색 결과 진단

HS 코드가 명시되지 않은 일반 규정은 코드별 정답과 직접 비교할 수 없습니다. 따라서 해당 문서가 각 검색 방식의 상위 10개 결과에 얼마나 포함되는지 별도로 확인합니다. 이 수치는 검색 빈도 진단이지 검색 정확도 지표가 아닙니다.


In [ ]:
general_summary = rankings.groupby('method').agg(top10_general_count=('is_code_specific',lambda x:int((~x).sum())),top10_total=('knowledge_id','count')).reset_index()
general_summary['general_share'] = general_summary['top10_general_count']/general_summary['top10_total']
display(general_summary)
display(rankings[rankings['rank']<=5].groupby(['method','source_sheet']).size().rename('n').reset_index())


## 7. 결과 저장

- `08_search_metrics_by_product.csv`: 제품별 검색 성능 지표
- `08_search_method_comparison.csv`: 검색 방식별 평균 및 표준편차
- `08_top10_retrieval_results.csv`: 제품별 검색 상위 10개 문서
- `08_fold_audit.csv`: Fold별 그룹 중복 점검
- `08_general_rule_diagnostics.csv`: 일반 규정 검색 비중 진단


In [ ]:
comparison_flat = results.groupby('method')[metric_cols].agg(['mean','std']).reset_index()
comparison_flat.columns = ['_'.join([str(x) for x in col if str(x)]) if isinstance(col,tuple) else col for col in comparison_flat.columns]
results.to_csv('08_search_metrics_by_product.csv',index=False,encoding='utf-8-sig')
comparison_flat.to_csv('08_search_method_comparison.csv',index=False,encoding='utf-8-sig')
rankings.to_csv('08_top10_retrieval_results.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('08_fold_audit.csv',index=False,encoding='utf-8-sig')
general_summary.to_csv('08_general_rule_diagnostics.csv',index=False,encoding='utf-8-sig')
print('08번 결과 저장 완료')
